# 🧠 AFFERENT URL Threat Detection -- Training Notebook v3

> **Dataset:** [URL-AF on Kaggle](https://www.kaggle.com/datasets/w5lasmbc/url-af)  
> **Architecture:** Cascade Tier 0 (Blocklist) -> Tier 1 (XGBoost) -> Tier 2 (CNN-BiLSTM) -> Stacker  
> **Classes:** 4-way `benign` / `phishing` / `malware` / `other`  
> **Split:** Domain-grouped zero-leakage (70 / 15 / 15)

| Section | Description |
|---|---|
| §0 | Setup & Data Loading |
| §1 | EDA & Distribution Analysis |
| §2 | Feature Engineering (~25 lexical features) |
| §3 | Baseline ML (LR, NB, SVM, DT) |
| §4 | Ensemble Baselines (RF standalone, XGBoost standalone) |
| §5 | Tier 1: XGBoost (Optuna-tuned) |
| §6 | Tier 2: CNN-BiLSTM (character-level, PyTorch) |
| §7 | Stacker (Logistic Regression meta-learner) |
| §8 | Cascade End-to-End Evaluation |
| §9 | Ablation Study |
| §10 | Segmented Evaluation + McNemar's Test |
| §11 | OOD: Indonesian Threat Case Study (judol + typosquat) |
| §12 | Model Export & Summary |

---
## §0 -- Setup & Data Loading

In [ ]:
!pip install -q tldextract optuna xgboost scikit-learn statsmodels

In [ ]:
import os, re, math, hashlib, warnings, time, json, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse

import tldextract, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder, label_binarize, MinMaxScaler
from sklearn.calibration import CalibratedClassifierCV
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | XGBoost: {xgb.__version__}")

In [ ]:
# ── Kaggle / Local dataset path ───────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/w5lasmbc/url-af
candidates = [
    Path("/kaggle/input/url-af"),
    Path("/kaggle/input/url-af/processed"),
    Path("../dataset/processed"),
    Path("dataset/processed"),
    Path("./"),
]
DATA_DIR = None
for c in candidates:
    if (c / "train.csv").exists():
        DATA_DIR = c
        break
if DATA_DIR is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("train.csv"))
    if found:
        DATA_DIR = found[0].parent
if DATA_DIR is None:
    DATA_DIR = Path("/kaggle/input/url-af")  # default Kaggle path

print(f"Detected DATA_DIR: {DATA_DIR}")

# Output directory: use /kaggle/working/afferent_models if on Kaggle, otherwise ./afferent_models
if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/afferent_models")
else:
    OUT_DIR = Path("./afferent_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {OUT_DIR}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
ood_df   = pd.read_csv(DATA_DIR / "indonesian_threat_casestudy.csv")

print(f"\nTrain : {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  OOD: {len(ood_df):,}")
print("Columns:", train_df.columns.tolist())
print("\nLabel dist (train):")
print(train_df["label"].value_counts())

In [ ]:
# ── SHA-256 integrity check (frozen baseline) ────────────────────
EXPECTED = {
    "train.csv":                      "78924F1F87BFF22D364FF6C311616B4F7950537329F27FED48C04405BA7432C2",
    "val.csv":                        "FE118A661CE6D4990B59DFC5182EECA9B7D675A30A268605BF01D2521C643017",
    "test.csv":                       "1BA4064401A49E6B874F3EFB055E9136A687BF5A786D446AC6CDE6EFF03F3348",
    "indonesian_threat_casestudy.csv":"ABF88658FE40530920BC4BB8FEE6887F2D504EB04B3876D8F758D9F78F0BDA10",
}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest().upper()

for fname, expected in EXPECTED.items():
    p = DATA_DIR / fname
    if not p.exists():
        print(f"  WARNING: {fname} not found -- skipping")
        continue
    ok = sha256_file(p) == expected
    status = "OK" if ok else "MISMATCH"
    print(f"  [{status}]  {fname}")

LABEL_ORDER = ["benign", "phishing", "malware", "other"]
le = LabelEncoder()
le.fit(LABEL_ORDER)
for df in [train_df, val_df, test_df]:
    df["label_enc"] = le.transform(df["label"])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_).tolist())))

---
## §1 -- EDA & Distribution Analysis

In [ ]:
def path_depth(url):
    try:
        parsed = urlparse(url if url.startswith("http") else "http://" + url)
        return len([p for p in parsed.path.split("/") if p])
    except:
        return 0

train_df["url_len"]    = train_df["url"].str.len()
train_df["path_depth"] = train_df["url"].apply(path_depth)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (df, name) in zip(axes, [(train_df,"Train"),(val_df,"Val"),(test_df,"Test")]):
    counts = df["label"].value_counts().reindex(LABEL_ORDER)
    bars = ax.bar(counts.index, counts.values,
                  color=["#2ecc71","#e74c3c","#e67e22","#9b59b6"])
    ax.set_title(f"{name} Split ({len(df):,})", fontweight="bold")
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                f"{v:,}", ha="center", fontsize=9)
plt.suptitle("Class Distribution per Split", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"eda_class_distribution.png", dpi=150)
plt.show()

print("\nMean URL length by class:")
print(train_df.groupby("label")["url_len"].mean().round(1))
print("\nMean path depth by class:")
print(train_df.groupby("label")["path_depth"].mean().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (col, xlim) in zip(axes, [("url_len",300),("path_depth",15)]):
    for label, color in zip(LABEL_ORDER, ["#2ecc71","#e74c3c","#e67e22","#9b59b6"]):
        data = train_df[train_df["label"]==label][col]
        ax.hist(data, bins=40, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(f"{col} Distribution by Class", fontweight="bold")
    ax.set_xlabel(col); ax.set_ylabel("Density"); ax.legend()
    ax.set_xlim(0, xlim)
plt.tight_layout()
plt.savefig(OUT_DIR/"eda_url_structure.png", dpi=150)
plt.show()

---
## §2 -- Feature Engineering

Lexical features (~25) extracted entirely from the URL string.  
External features (domain_age, SSL, ASN, redirect) are **stubs returning -1** for offline Kaggle training.  
In production, these are resolved with a 300ms-timeout cache-backed lookup.

In [ ]:
BRANDS = [
    "paypal","facebook","google","apple","amazon","microsoft",
    "netflix","instagram","twitter","linkedin","dropbox",
    # Indonesian
    "klikbca","mandiri","bri","bni","permata","cimb",
    "tokopedia","shopee","gojek","grab","traveloka",
]
HIGH_RISK_TLDS = {
    ".xyz",".top",".club",".info",".online",".site",".live",
    ".icu",".tk",".ml",".ga",".cf",".gq",".pw",".cc",
    ".buzz",".fun",".click",".link",".loan",".work",".win",
}
SUSPICIOUS_KW = [
    "login","signin","verify","secure","account","update",
    "confirm","banking","wallet","password","credential",
    "webscr","cmd=_","click","redirect",
    # Indonesian
    "daftar","masuk","konfirmasi","slot","bonus","jackpot",
]

def levenshtein(s1, s2):
    if len(s1) < len(s2): s1, s2 = s2, s1
    prev = list(range(len(s2)+1))
    for c1 in s1:
        curr = [prev[0]+1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j+1]+1, curr[-1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

def shannon_entropy(s):
    if not s: return 0.0
    cnt = Counter(s); n = len(s)
    return -sum((v/n)*math.log2(v/n) for v in cnt.values())

def extract_features(url: str) -> dict:
    url = str(url)
    try:
        parsed = urlparse(url if url.startswith("http") else "http://"+url)
    except:
        parsed = urlparse("http://unknown")
    ext       = tldextract.extract(url)
    domain    = ext.domain or ""
    subdomain = ext.subdomain or ""
    suffix    = ext.suffix or ""
    path      = parsed.path or ""
    query     = parsed.query or ""
    host      = parsed.netloc.split(":")[0]
    path_parts = [p for p in path.split("/") if p]
    is_ip = bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))
    lev_min = min((levenshtein(domain.lower(), b) for b in BRANDS), default=99)
    url_lower = url.lower()
    return {
        "url_len"        : len(url),
        "domain_len"     : len(host),
        "path_len"       : len(path),
        "query_len"      : len(query),
        "path_depth"     : len(path_parts),
        "query_param_cnt": len(query.split("&")) if query else 0,
        "num_digits"     : sum(c.isdigit() for c in url),
        "num_dashes"     : url.count("-"),
        "num_dots"       : url.count("."),
        "num_slashes"    : url.count("/"),
        "num_special"    : sum(c in "@?=&%+#" for c in url),
        "num_upper"      : sum(c.isupper() for c in url),
        "subdomain_count": len([s for s in subdomain.split(".") if s]),
        "has_at"         : int("@" in url),
        "has_port"       : int(":" in host),
        "uses_https"     : int(parsed.scheme=="https"),
        "is_ip"          : int(is_ip),
        "tld_risk"       : float(f".{suffix}".lower() in HIGH_RISK_TLDS),
        "levenshtein_min": lev_min,
        "keyword_count"  : sum(1 for kw in SUSPICIOUS_KW if kw in url_lower),
        "entropy_domain" : shannon_entropy(domain),
        "entropy_url"    : shannon_entropy(url),
        "has_brand_name" : int(any(b in url_lower for b in BRANDS)),
        "ratio_digits"   : sum(c.isdigit() for c in url) / max(len(url),1),
        "ratio_dashes"   : url.count("-") / max(len(host),1),
        "typosquat_subs" : len(re.findall(r"[0-9](?=[a-z])|[a-z](?=[0-9])", domain)),
        # External stubs (offline training -- use -1 as missing indicator)
        "domain_age_days": -1,
        "ssl_valid"      : -1,
        "ssl_trust_score": -1,
        "redirect_count" : -1,
        "asn_reputation" : -1,
    }

# Quick sanity check
print(pd.Series(extract_features("http://kl1kbca-secure.xyz/login/verify?id=12345")))

In [ ]:
print("Extracting features for all splits...")
t0 = time.time()

def extract_df(df):
    return pd.DataFrame(df["url"].apply(extract_features).tolist(), index=df.index)

X_train_lex = extract_df(train_df); print(f"  train: {time.time()-t0:.1f}s")
X_val_lex   = extract_df(val_df);   print(f"  val:   {time.time()-t0:.1f}s")
X_test_lex  = extract_df(test_df);  print(f"  test:  {time.time()-t0:.1f}s")
X_ood_lex   = extract_df(ood_df);   print(f"  ood:   {time.time()-t0:.1f}s  [total]")

# Columns excluding external stubs (for baseline models)
LEX_COLS = [c for c in X_train_lex.columns
            if c not in {"domain_age_days","ssl_valid","ssl_trust_score",
                         "redirect_count","asn_reputation"}]
ALL_COLS = X_train_lex.columns.tolist()
print(f"\nLexical features: {len(LEX_COLS)}  |  All features: {len(ALL_COLS)}")

y_train = train_df["label_enc"].values
y_val   = val_df["label_enc"].values
y_test  = test_df["label_enc"].values

In [ ]:
# Feature-label correlation heatmap
corr = X_train_lex[LEX_COLS].copy()
corr["label"] = y_train
c = corr.corr()["label"].drop("label").sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(c.index, c.values, color=["#e74c3c" if v<0 else "#2ecc71" for v in c.values])
ax.set_title("Feature Correlation with Label (0=benign)", fontweight="bold")
ax.axvline(0, color="white", lw=0.8)
plt.tight_layout()
plt.savefig(OUT_DIR/"feature_correlations.png", dpi=150)
plt.show()

---
## §3 -- Baseline ML Models

In [ ]:
def evaluate(name, y_true, y_pred, y_proba=None):
    acc  = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average="weighted")
    f1_m = f1_score(y_true, y_pred, average="macro")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)
    fp   = cm.sum(axis=0) - np.diag(cm)
    fn   = cm.sum(axis=1) - np.diag(cm)
    tn   = cm.sum() - (fp + fn + np.diag(cm))
    mean_fpr = (fp / (fp + tn + 1e-9)).mean()

    r = {"Model": name, "Accuracy": acc, "F1-weighted": f1_w,
         "F1-macro": f1_m, "Precision": prec, "Recall": rec, "FPR": mean_fpr}
    if y_proba is not None:
        y_bin = label_binarize(y_true, classes=list(range(4)))
        r["ROC-AUC"] = roc_auc_score(y_bin, y_proba, multi_class="ovr", average="macro")
        r["PR-AUC"]  = average_precision_score(y_bin, y_proba, average="macro")
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"  Acc: {acc:.4f}  F1-w: {f1_w:.4f}  F1-m: {f1_m:.4f}")
    print(f"  Prec: {prec:.4f}  Rec: {rec:.4f}  FPR: {mean_fpr:.4f}")
    if "PR-AUC" in r:
        print(f"  ROC-AUC: {r['ROC-AUC']:.4f}  PR-AUC: {r['PR-AUC']:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return r

RESULTS = []
X_tr = X_train_lex[LEX_COLS].fillna(-1).values
X_vl = X_val_lex[LEX_COLS].fillna(-1).values
X_te = X_test_lex[LEX_COLS].fillna(-1).values
print("Feature matrix shape:", X_tr.shape)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, n_jobs=-1)
lr.fit(X_tr, y_train)
RESULTS.append(evaluate("Logistic Regression", y_test, lr.predict(X_te), lr.predict_proba(X_te)))

In [ ]:
# Naive Bayes
scaler_nb = MinMaxScaler()
nb = GaussianNB()
nb.fit(scaler_nb.fit_transform(X_tr), y_train)
p = nb.predict_proba(scaler_nb.transform(X_te))
RESULTS.append(evaluate("Naive Bayes", y_test, nb.predict(scaler_nb.transform(X_te)), p))

In [ ]:
# SVM (LinearSVC + calibration for probabilities)
svm = CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0, random_state=SEED), cv=3)
svm.fit(X_tr, y_train)
RESULTS.append(evaluate("SVM (LinearSVC calibrated)", y_test, svm.predict(X_te), svm.predict_proba(X_te)))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=15, min_samples_leaf=5, random_state=SEED)
dt.fit(X_tr, y_train)
RESULTS.append(evaluate("Decision Tree", y_test, dt.predict(X_te), dt.predict_proba(X_te)))

---
## §4 -- Ensemble Baselines (RF standalone, XGBoost standalone)

In [ ]:
# Random Forest standalone (NOT in cascade -- dropped in v3, kept here as baseline)
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=3,
                             n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_train)
RESULTS.append(evaluate("Random Forest (standalone)", y_test, rf.predict(X_te), rf.predict_proba(X_te)))

In [ ]:
# XGBoost standalone (lexical-only, no hyperparameter tuning)
dtrain = xgb.DMatrix(X_tr, label=y_train)
dval   = xgb.DMatrix(X_vl, label=y_val)
dtest  = xgb.DMatrix(X_te, label=y_test)

xgb_params_base = {
    "objective": "multi:softprob", "num_class": 4,
    "eval_metric": ["mlogloss","merror"],
    "eta": 0.1, "max_depth": 6, "subsample": 0.8,
    "colsample_bytree": 0.8, "seed": SEED,
    "device": "cuda" if DEVICE=="cuda" else "cpu",
    "verbosity": 0,
}
xgb_base = xgb.train(xgb_params_base, dtrain, num_boost_round=300,
                     evals=[(dval,"val")], early_stopping_rounds=20, verbose_eval=50)
p_xgb_base = xgb_base.predict(dtest).reshape(-1,4)
RESULTS.append(evaluate("XGBoost Standalone (lexical-only)", y_test, p_xgb_base.argmax(1), p_xgb_base))

---
## §5 -- Tier 1: XGBoost (Optuna-tuned, maximize PR-AUC on val)

In [ ]:
def xgb_objective(trial):
    params = {
        "objective": "multi:softprob", "num_class": 4, "eval_metric": "mlogloss",
        "eta":             trial.suggest_float("eta", 0.03, 0.3, log=True),
        "max_depth":       trial.suggest_int("max_depth", 4, 10),
        "min_child_weight":trial.suggest_int("min_child_weight", 1, 10),
        "subsample":       trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":           trial.suggest_float("gamma", 0.0, 5.0),
        "lambda":          trial.suggest_float("lambda", 0.1, 10.0, log=True),
        "alpha":           trial.suggest_float("alpha", 0.0, 5.0),
        "seed": SEED, "verbosity": 0,
        "device": "cuda" if DEVICE=="cuda" else "cpu",
    }
    m = xgb.train(params, dtrain, num_boost_round=500,
                  evals=[(dval,"val")], early_stopping_rounds=30, verbose_eval=False)
    p = m.predict(dval).reshape(-1,4)
    y_bin = label_binarize(y_val, classes=list(range(4)))
    return average_precision_score(y_bin, p, average="macro")

study = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, n_trials=40, show_progress_bar=True)
print(f"Best PR-AUC (val): {study.best_value:.4f}")
print("Best params:", study.best_params)

In [ ]:
best = study.best_params
best.update({"objective":"multi:softprob","num_class":4,
             "eval_metric":["mlogloss","merror"],
             "seed":SEED,
             "device":"cuda" if DEVICE=="cuda" else "cpu"})

tier1_xgb = xgb.train(best, dtrain, num_boost_round=800,
                      evals=[(dval,"val")], early_stopping_rounds=30, verbose_eval=50)
tier1_xgb.save_model(str(OUT_DIR/"tier1_xgboost.json"))
print("Tier-1 XGBoost saved.")

p_t1_test = tier1_xgb.predict(dtest).reshape(-1,4)
pred_t1   = p_t1_test.argmax(1)
res_t1    = evaluate("Tier-1 XGBoost (tuned + lexical)", y_test, pred_t1, p_t1_test)
RESULTS.append(res_t1)

p_t1_val = tier1_xgb.predict(dval).reshape(-1,4)  # keep for stacker

In [ ]:
# Feature importance
fi = pd.DataFrame({"feat":list(tier1_xgb.get_score("gain").keys()),
                   "imp":list(tier1_xgb.get_score("gain").values())})
fi = fi.sort_values("imp",ascending=True).tail(25)

fig, ax = plt.subplots(figsize=(10,7))
ax.barh(fi["feat"], fi["imp"], color="#3498db")
ax.set_title("XGBoost Tier-1 -- Feature Importance (Gain)", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"tier1_feature_importance.png", dpi=150)
plt.show()

---
## §6 -- Tier 2: CNN-BiLSTM (Character-Level, PyTorch)

Architecture: `Embedding -> Multi-scale CNN (k=3,5,7) -> BiLSTM (2 layers) -> Attention Pooling -> 4-class head`

In [ ]:
URL_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    r".-_~:/?#[]@!$&'()*+,;=%"
)
CHAR2IDX  = {c: i+2 for i, c in enumerate(URL_CHARS)}  # 0=PAD, 1=UNK
VOCAB_SIZE = len(CHAR2IDX) + 2
MAX_LEN    = 200

def url_to_tensor(url):
    ids = [CHAR2IDX.get(c, 1) for c in str(url)[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

class URLDataset(Dataset):
    def __init__(self, urls, labels):
        self.urls = urls.reset_index(drop=True) if hasattr(urls,"reset_index") else pd.Series(list(urls)).reset_index(drop=True)
        self.labels = labels
    def __len__(self): return len(self.urls)
    def __getitem__(self, i): return url_to_tensor(self.urls[i]), int(self.labels[i])

BS = 256
train_ds = URLDataset(train_df["url"], y_train)
val_ds   = URLDataset(val_df["url"],   y_val)
test_ds  = URLDataset(test_df["url"],  y_test)
ood_ds   = URLDataset(ood_df["url"],   np.zeros(len(ood_df),int))

def mk_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BS, shuffle=shuffle, num_workers=2, pin_memory=DEVICE=="cuda")

train_loader = mk_loader(train_ds, True)
val_loader   = mk_loader(val_ds,   False)
test_loader  = mk_loader(test_ds,  False)
ood_loader   = mk_loader(ood_ds,   False)
print(f"Vocab size: {VOCAB_SIZE}  Max URL length: {MAX_LEN}")

In [ ]:
class CNNBiLSTM(nn.Module):
    # Character-level CNN-BiLSTM with attention pooling for URL threat detection.
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filters=128,
                 kernels=(3,5,7), lstm_h=128, lstm_l=2, n_cls=4, drop=0.4):
        super().__init__()
        self.embed  = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs  = nn.ModuleList([
            nn.Sequential(nn.Conv1d(emb, filters, k, padding=k//2),
                          nn.GELU(), nn.Dropout(drop/2))
            for k in kernels])
        cnn_dim = filters * len(kernels)
        self.bilstm = nn.LSTM(cnn_dim, lstm_h, lstm_l,
                              bidirectional=True, batch_first=True,
                              dropout=drop if lstm_l>1 else 0.0)
        self.attn = nn.Linear(lstm_h*2, 1)
        self.head = nn.Sequential(
            nn.Linear(lstm_h*2, 128), nn.GELU(), nn.Dropout(drop),
            nn.Linear(128, n_cls)
        )
    def forward(self, x):
        e  = self.embed(x)                                         # B,L,E
        ci = e.permute(0,2,1)                                      # B,E,L
        c  = torch.cat([cv(ci).permute(0,2,1) for cv in self.convs], -1)  # B,L,Fc
        out,_ = self.bilstm(c)                                     # B,L,H*2
        w  = torch.softmax(self.attn(out), 1)                      # B,L,1
        p  = (out*w).sum(1)                                        # B,H*2
        return self.head(p)

model_dl = CNNBiLSTM().to(DEVICE)
total = sum(p.numel() for p in model_dl.parameters())
print(f"CNN-BiLSTM parameters: {total:,}")

In [ ]:
def train_dl(model, train_loader, val_loader, epochs=15, lr=3e-4):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_vl, best_state = float("inf"), None
    tr_hist, vl_hist = [], []

    for ep in range(1, epochs+1):
        model.train(); tl = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(xb)
        model.eval(); vl = 0.0; correct = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits  = model(xb)
                vl     += crit(logits, yb).item() * len(xb)
                correct += (logits.argmax(1)==yb).sum().item()
        tl /= len(train_loader.dataset); vl /= len(val_loader.dataset)
        va  = correct / len(val_loader.dataset)
        tr_hist.append(tl); vl_hist.append(vl)
        sched.step()
        if vl < best_vl:
            best_vl = vl
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        print(f"  Epoch {ep:02d}/{epochs} | TrLoss {tl:.4f} | ValLoss {vl:.4f} | ValAcc {va:.4f}"
              + (" [best]" if vl==best_vl else ""))
    model.load_state_dict(best_state)
    return tr_hist, vl_hist

print("Training CNN-BiLSTM...")
tr_hist, vl_hist = train_dl(model_dl, train_loader, val_loader, epochs=15)

In [ ]:
# Training curve
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(tr_hist, label="Train Loss", color="#3498db")
ax.plot(vl_hist, label="Val Loss",   color="#e74c3c")
ax.set(xlabel="Epoch", ylabel="Cross-Entropy Loss",
       title="CNN-BiLSTM Training Curve")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/"tier2_training_curve.png", dpi=150); plt.show()

In [ ]:
def dl_infer(model, loader):
    model.eval(); probas, preds = [], []
    with torch.no_grad():
        for xb, _ in loader:
            p = F.softmax(model(xb.to(DEVICE)),1).cpu().numpy()
            probas.append(p); preds.extend(p.argmax(1))
    return np.vstack(probas), np.array(preds)

p_t2_test, pred_t2 = dl_infer(model_dl, test_loader)
res_t2 = evaluate("Tier-2 CNN-BiLSTM (standalone)", y_test, pred_t2, p_t2_test)
RESULTS.append(res_t2)
torch.save(model_dl.state_dict(), str(OUT_DIR/"tier2_cnnbilstm.pt"))
print("Tier-2 CNN-BiLSTM saved.")

p_t2_val, _ = dl_infer(model_dl, val_loader)  # keep for stacker

# Confusion matrix
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(confusion_matrix(y_test, pred_t2), annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set(title="CNN-BiLSTM Confusion Matrix (Test)",
       ylabel="True", xlabel="Predicted")
plt.tight_layout()
plt.savefig(OUT_DIR/"tier2_confusion_matrix.png", dpi=150); plt.show()

---
## §7 -- Stacker (Logistic Regression Meta-Learner)

Trained on **validation set** predicted probabilities from Tier-1 and Tier-2.  
Input: `[xgb_proba(4), dl_proba(4)]` -> 8-dim -> calibrated 4-class confidence.

In [ ]:
X_stack_val  = np.hstack([p_t1_val, p_t2_val])    # (N_val, 8)
X_stack_test = np.hstack([p_t1_test, p_t2_test])  # (N_test, 8)

stacker = LogisticRegression(C=1.0, max_iter=500, random_state=SEED, n_jobs=-1)
stacker.fit(X_stack_val, y_val)

p_stack = stacker.predict_proba(X_stack_test)
pred_stack = stacker.predict(X_stack_test)
res_stack = evaluate("Stacker (LR meta Tier1+Tier2)", y_test, pred_stack, p_stack)
RESULTS.append(res_stack)

with open(OUT_DIR/"stacker.pkl", "wb") as f: pickle.dump(stacker, f)
print("Stacker saved.")

---
## §8 -- Cascade System: End-to-End Evaluation

**Tier 0** -> Tier 1 (confidence gate +-0.85/0.15) -> **Tier 2** (ambiguous only) -> Stacker -> Hard override rule.

In [ ]:
CONF_HIGH = 0.85
CONF_LOW  = 0.15

def cascade_predict(urls_series, feats_df, xgb_m, dl_m, stacker_m):
    n = len(urls_series)
    # Tier 1
    dm      = xgb.DMatrix(feats_df[LEX_COLS].fillna(-1).values)
    xgb_p   = xgb_m.predict(dm).reshape(-1,4)
    xgb_c   = xgb_p.max(1)
    ambig   = (xgb_c > CONF_LOW) & (xgb_c < CONF_HIGH)

    # Tier 2 (ambiguous only)
    dl_p_full = np.zeros((n,4))
    if ambig.any():
        ambig_urls = urls_series.reset_index(drop=True)[ambig].reset_index(drop=True)
        ambig_lbl  = np.zeros(ambig.sum(), int)
        ambig_ds   = URLDataset(ambig_urls, ambig_lbl)
        ambig_ld   = mk_loader(ambig_ds, False)
        dl_p_ambig, _ = dl_infer(dl_m, ambig_ld)
        dl_p_full[ambig] = dl_p_ambig

    # Stacker for ambiguous
    final_p = xgb_p.copy()
    if ambig.any():
        meta = np.hstack([xgb_p[ambig], dl_p_full[ambig]])
        final_p[ambig] = stacker_m.predict_proba(meta)

    # Hard override: domain_age_days < 3 AND levenshtein_min <= 2
    # (Won't fire on offline test since domain_age stubs are -1)
    override = (
        (feats_df["domain_age_days"].values != -1) &
        (feats_df["domain_age_days"].values < 3) &
        (feats_df["levenshtein_min"].values <= 2)
    )
    if override.any():
        phish_idx = list(le.classes_).index("phishing")
        final_p[override, phish_idx] = np.maximum(final_p[override, phish_idx], 0.70)

    return final_p.argmax(1), final_p, np.where(ambig, 2, 1).astype(int), ambig

print("Running cascade inference on test set...")
pred_casc, p_casc, tier_used, ambig_mask = cascade_predict(
    test_df["url"], X_test_lex, tier1_xgb, model_dl, stacker
)
print(f"Tier 1 only : {(tier_used==1).sum():,} ({(tier_used==1).mean()*100:.1f}%)")
print(f"Tier 2 used : {(tier_used==2).sum():,} ({(tier_used==2).mean()*100:.1f}%)")

In [ ]:
res_casc = evaluate("Cascade System (Tier 0+1+2 + Stacker)", y_test, pred_casc, p_casc)
RESULTS.append(res_casc)

# Routing pie
fig, ax = plt.subplots(figsize=(6,6))
t1c, t2c = (tier_used==1).sum(), (tier_used==2).sum()
ax.pie([t1c, t2c],
       labels=[f"Tier 1 only\n{t1c:,}", f"Tier 1+2\n{t2c:,}"],
       colors=["#3498db","#e74c3c"], autopct="%1.1f%%", startangle=90,
       wedgeprops={"edgecolor":"white","linewidth":2})
ax.set_title("Cascade Routing Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"cascade_routing.png", dpi=150); plt.show()

---
## §9 -- Ablation Study

In [ ]:
# CNN-only and LSTM-only for DL ablation
class CNNOnly(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filt=256, n_cls=4, drop=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb,filt,k,padding=k//2) for k in [3,5,7]])
        self.head  = nn.Sequential(nn.Dropout(drop),nn.Linear(filt*3,128),
                                   nn.GELU(),nn.Dropout(drop),nn.Linear(128,n_cls))
    def forward(self,x):
        e = self.embed(x).permute(0,2,1)
        p = torch.cat([F.adaptive_max_pool1d(F.gelu(c(e)),1).squeeze(-1) for c in self.convs],1)
        return self.head(p)

class LSTMOnly(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, emb=64, h=256, n_cls=4, drop=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm  = nn.LSTM(emb,h,2,bidirectional=True,batch_first=True,dropout=drop)
        self.head  = nn.Sequential(nn.Dropout(drop),nn.Linear(h*2,n_cls))
    def forward(self,x):
        _,(h,_) = self.lstm(self.embed(x))
        return self.head(torch.cat([h[-1],h[-2]],1))

for name, m in [("CNN-only", CNNOnly()), ("LSTM-only", LSTMOnly())]:
    m = m.to(DEVICE)
    print(f"\nTraining {name}...")
    train_dl(m, train_loader, val_loader, epochs=10)
    p, pred = dl_infer(m, test_loader)
    RESULTS.append(evaluate(f"Ablation: {name}", y_test, pred, p))

In [ ]:
# Summarize ablation: cascade vs single models
ablation_rows = [
    ("XGBoost Tier-1",          res_t1),
    ("CNN-BiLSTM Tier-2",       res_t2),
    ("Stacker (T1+T2)",         res_stack),
    ("Cascade System",          res_casc),
]
metrics = ["F1-weighted","PR-AUC","FPR"]
fig, axes = plt.subplots(1,3,figsize=(15,5))
colors = ["#3498db","#2ecc71","#9b59b6","#e67e22"]
for ax, met in zip(axes, metrics):
    vals = [r[1].get(met, 0) for r in ablation_rows]
    labs = [r[0] for r in ablation_rows]
    bars = ax.bar(labs, vals, color=colors)
    ax.set_title(met, fontweight="bold")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Ablation: Model Architecture Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"ablation_comparison.png", dpi=150); plt.show()

---
## §10 -- Segmented Evaluation + McNemar's Statistical Test

In [ ]:
# Full results table
res_df = pd.DataFrame(RESULTS).set_index("Model").round(4)
print("\n" + "="*70)
print("COMPREHENSIVE MODEL COMPARISON -- TEST SET")
print("="*70)
print(res_df.to_string())
res_df.to_csv(OUT_DIR/"all_model_results.csv")

In [ ]:
# Heatmap
plot_cols = [c for c in ["Accuracy","F1-weighted","F1-macro","PR-AUC","FPR"] if c in res_df.columns]
fig, ax = plt.subplots(figsize=(12, len(res_df)*0.55+2))
sns.heatmap(res_df[plot_cols].astype(float), annot=True, fmt=".4f",
            cmap="RdYlGn", linewidths=0.3, ax=ax)
ax.set_title("All Models -- Test Set Performance", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR/"all_models_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Per-class PR-AUC breakdown
print("Per-class Evaluation -- Cascade System:")
y_bin = label_binarize(y_test, classes=list(range(4)))
for i, cls in enumerate(le.classes_):
    pr_auc = average_precision_score(y_bin[:,i], p_casc[:,i])
    fp_i = ((pred_casc==i) & (y_test!=i)).sum()
    tn_i = ((pred_casc!=i) & (y_test!=i)).sum()
    fpr_i = fp_i / (fp_i+tn_i+1e-9)
    n_cls = (y_test==i).sum()
    print(f"  [{cls:8s}] n={n_cls:5,}  PR-AUC={pr_auc:.4f}  FPR={fpr_i:.4f}")

In [ ]:
# McNemar's test: Cascade vs Tier-1 XGBoost
c_c = (pred_casc==y_test)
c_x = (pred_t1==y_test)
n11 = ( c_c &  c_x).sum(); n10 = ( c_c & ~c_x).sum()
n01 = (~c_c &  c_x).sum(); n00 = (~c_c & ~c_x).sum()
tbl = [[n11,n10],[n01,n00]]
result = mcnemar_test(tbl, exact=False, correction=True)
print("McNemar's Test -- Cascade vs XGBoost Tier-1")
print(f"  Both correct:  {n11:,}")
print(f"  Only Cascade:  {n10:,}")
print(f"  Only XGBoost:  {n01:,}")
print(f"  Both wrong:    {n00:,}")
print(f"  Chi2: {result.statistic:.4f}  p-value: {result.pvalue:.6f}")
if result.pvalue < 0.05:
    print("  -> Statistically significant difference (p < 0.05)")
else:
    print("  -> NOT statistically significant (p >= 0.05)")

In [ ]:
# Per-source breakdown
if "source" in test_df.columns:
    print("\nPer-source F1 (Cascade):")
    for src in sorted(test_df["source"].unique()):
        mask = test_df["source"].values == src
        if mask.sum() < 10: continue
        f1 = f1_score(y_test[mask], pred_casc[mask], average="weighted", zero_division=0)
        print(f"  [{src:30s}]  n={mask.sum():6,}  F1={f1:.4f}")

---
## §11 -- OOD Evaluation: Indonesian Threat Case Study

**Zero-shot** generalization test on 299 judol (online gambling) & typosquatting URLs  
targeting Indonesian banks -- **never seen during training**.

In [ ]:
print("OOD sample preview:")
print(ood_df.head(10))
print("\nOOD label distribution:")
if "label" in ood_df.columns: print(ood_df["label"].value_counts())

In [ ]:
# Tier-1
d_ood = xgb.DMatrix(X_ood_lex[LEX_COLS].fillna(-1).values)
p_t1_ood = tier1_xgb.predict(d_ood).reshape(-1,4)

# Tier-2 (run all OOD through DL for analysis)
p_t2_ood, _ = dl_infer(model_dl, ood_loader)

# Stacker
p_stack_ood = stacker.predict_proba(np.hstack([p_t1_ood, p_t2_ood]))
pred_ood    = p_stack_ood.argmax(1)
conf_ood    = p_stack_ood.max(1)
label_ood   = le.inverse_transform(pred_ood)

print("OOD Predicted class distribution:")
print(pd.Series(label_ood).value_counts())
print(f"\nMean confidence : {conf_ood.mean():.4f}")
print(f"High conf (>=0.85): {(conf_ood>=0.85).mean()*100:.1f}%  (would auto-block)")
print(f"Review zone (0.6-0.85): {((conf_ood>=0.6)&(conf_ood<0.85)).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))

# Confidence histogram
ax = axes[0]
ax.hist(conf_ood, bins=25, color="#e74c3c", alpha=0.85, edgecolor="white")
ax.axvline(0.85, color="white",  lw=2, ls="--", label="Block threshold (0.85)")
ax.axvline(0.60, color="yellow", lw=2, ls="--", label="Review threshold (0.60)")
ax.set(title="OOD Confidence Distribution\n(Indonesian Threats -- Zero-Shot)",
       xlabel="Model Confidence", ylabel="Count")
ax.legend()

# Predicted class bars
ax = axes[1]
counts = pd.Series(label_ood).value_counts().reindex(LABEL_ORDER, fill_value=0)
bars = ax.bar(counts.index, counts.values, color=["#2ecc71","#e74c3c","#e67e22","#9b59b6"])
for bar, v in zip(bars, counts.values):
    if v > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f"{v}\n({v/len(label_ood)*100:.1f}%)", ha="center", fontsize=10)
ax.set(title="OOD Predicted Classes (Zero-Shot)", ylabel="Count")

plt.suptitle("OOD Generalization: Indonesian Judol & Bank Typosquat URLs",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"ood_evaluation.png", dpi=150); plt.show()

In [ ]:
# If ground-truth labels exist
if "is_malicious" in ood_df.columns:
    y_ood_bin   = ood_df["is_malicious"].values.astype(int)
    benign_idx  = list(le.classes_).index("benign")
    detected    = (pred_ood != benign_idx).astype(int)
    detection_r = (detected == y_ood_bin).mean()
    recall_thr  = ((detected==1)&(y_ood_bin==1)).sum() / max(y_ood_bin.sum(),1)
    print(f"OOD Detection Rate : {detection_r:.4f}")
    print(f"OOD Threat Recall  : {recall_thr:.4f} (zero-shot)")
else:
    top = pd.DataFrame({"url":ood_df["url"].values, "predicted":label_ood,
                        "confidence":conf_ood}).sort_values("confidence",ascending=False)
    print("Top 15 OOD predictions by confidence:")
    print(top.head(15).to_string(index=False))

---
## §12 -- Model Export & Final Summary

In [ ]:
# Label encoder
with open(OUT_DIR/"label_encoder.json","w") as f:
    json.dump({"classes":le.classes_.tolist(),
               "mapping":dict(zip(le.classes_,le.transform(le.classes_).tolist()))}, f)

# Feature columns
with open(OUT_DIR/"feature_cols.json","w") as f:
    json.dump({"lexical_cols":LEX_COLS, "all_cols":ALL_COLS}, f)

# Cascade config
cfg = {
    "tier1_confident_high": CONF_HIGH,
    "tier1_confident_low":  CONF_LOW,
    "action_thresholds": {"block_auto":0.85,"block_review":0.60,"allow_log":0.0},
    "hard_override": {"domain_age_days_lt":3, "levenshtein_min_lte":2},
    "vocab_size": VOCAB_SIZE, "max_url_len": MAX_LEN,
    "char2idx": CHAR2IDX, "label_order": LABEL_ORDER,
}
with open(OUT_DIR/"cascade_config.json","w") as f: json.dump(cfg, f, indent=2)

# Full results
res_df.to_csv(OUT_DIR/"results_full.csv")

print("All artifacts saved to:", OUT_DIR)
for fp in sorted(OUT_DIR.iterdir()):
    print(f"  {fp.name:42s} {fp.stat().st_size/1024:8.1f} KB")

In [ ]:
print("\n" + "="*70)
print("AFFERENT v3 -- FINAL TRAINING SUMMARY")
print("="*70)
key = ["Logistic Regression","Random Forest (standalone)",
       "XGBoost Standalone (lexical-only)","Tier-1 XGBoost (tuned + lexical)",
       "Tier-2 CNN-BiLSTM (standalone)","Stacker (LR meta Tier1+Tier2)",
       "Cascade System (Tier 0+1+2 + Stacker)"]
summary = res_df.loc[[m for m in key if m in res_df.index],
                     [c for c in ["Accuracy","F1-weighted","PR-AUC","FPR"] if c in res_df.columns]]
print(summary.to_string())
print(f"\nCascade routing:")
print(f"  Tier 1 only : {(tier_used==1).sum():,} ({(tier_used==1).mean()*100:.1f}%)")
print(f"  Tier 1+2    : {(tier_used==2).sum():,} ({(tier_used==2).mean()*100:.1f}%)")
print(f"\nOOD (Indonesian threats) -- {len(ood_df)} zero-shot URLs")
print(f"  Predicted malicious : {(pred_ood != list(le.classes_).index('benign')).sum()}")
print("="*70)